In [0]:
import os
import pandas as pd

from multiprocessing.pool import ThreadPool
from functools import partial
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, LongType, DoubleType, TimestampType, BooleanType

In [0]:
schema_agg_transacoes_diarias = StructType([
    StructField("SK_AGG_TRANSACOES_DIARIAS", StringType(), True),
    StructField("FK_CONTA", LongType(), True),
    StructField("FK_CLIENTE", LongType(), True),
    StructField("FK_AGENCIA", LongType(), True),
    StructField("FK_COLABORADOR", LongType(), True),
    StructField("DATA_COMPLETA", StringType(), True),
    StructField("TIPO_TRASACAO", StringType(), True),
    StructField("TRANSACAO_TOTAL", DoubleType(), True)
])

schema_dim_agencias = StructType([
    StructField("PK_AGENCIA", IntegerType(), True),
    StructField("NUMERO_AGENCIA", IntegerType(), True),
    StructField("NOME_AGENCIA", StringType(), True),
    StructField("ENDERECO_AGENCIA", StringType(), True),
    StructField("CIDADE_AGENCIA", StringType(), True),
    StructField("UF_AGENCIA", StringType(), True), 
    StructField("DATA_ABERTURA_AGENCIA", DateType(), True),       
    StructField("TIPO_AGENCIA", StringType(), True)
])

schema_dim_clientes = StructType([
    StructField("PK_CLIENTE", LongType(), True),
    StructField("NOME_CLIENTE", StringType(), True),
    StructField("EMAIL_CLIENTE", StringType(), True),
    StructField("TIPO_CLIENTE", StringType(), True),
    StructField("CPFCNPJ_CLIENTE", StringType(), True),
    StructField("TS_INCLUSAO", TimestampType(), True),
    StructField("DATA_NASCIMENTO_CLEINTE", DateType(), True),
    StructField("ENDERECO_CLIENTE", StringType(), True),
    StructField("CEP_CLIENTE", StringType(), True),
    StructField("UF_CLIENTE", StringType(), True)
])

schema_dim_colaboradores = StructType([
    StructField("PK_COLABORADOR", LongType(), True),
    StructField("NOME_COLABORADOR", StringType(), True),
    StructField("EMAIL_COLABORADOR", StringType(), True),
    StructField("CPF_COLABORADOR", StringType(), True),
    StructField("DATA_NASCIMENTO_COLABORADOR", DateType(), True),
    StructField("ENDERECO_COLABORADOR", StringType(), True),
    StructField("CEP_COLABORADOR", StringType(), True),
    StructField("UF_COLABORADOR", StringType(), True)
])

schema_dim_contas = StructType([
    StructField("PK_CONTA", LongType(), True),
    StructField("NUMERO_CONTA", LongType(), True),
    StructField("TIPO_CONTA", StringType(), True),
    StructField("TS_ABERTURA_CONTA", TimestampType(), True)
])

schema_dim_datas = StructType([
    StructField("PK_DATA", DateType(), True),
    StructField("ANO", IntegerType(), True),
    StructField("MES", IntegerType(), True),
    StructField("DIA", IntegerType(), True),
    StructField("TRIMESTRE", IntegerType(), True),
    StructField("DATA_COMPLETA", StringType(), True),
    StructField("DIA_DA_SEMANA", IntegerType(), True),
    StructField("IS_WEEKEND", BooleanType(), True)
])

schema_fct_contas = StructType([
    StructField("PK_CONTA", LongType(), True),
    StructField("FK_CLIENTE", LongType(), True),
    StructField("FK_AGENCIA", LongType(), True),
    StructField("FK_COLABORADOR", LongType(), True),
    StructField("SALDO_TOTAL", DoubleType(), True),
    StructField("SALDO_DISPONIVEL", DoubleType(), True),
    StructField("TS_ULTIMO_LANCAMENTO", TimestampType(), True)
])

schema_fct_propostas_de_credito = StructType([
    StructField("PK_PROPOSTA", LongType(), True),
    StructField("FK_CLIENTE", LongType(), True),
    StructField("FK_COLABORADOR", LongType(), True),
    StructField("FK_AGENCIA", LongType(), True),
    StructField("NUMERO_PROPOSTA", LongType(), True),
    StructField("TS_ENTRADA_PROPOSTA", TimestampType(), True),
    StructField("TAXA_JUROS_MENSAL", DoubleType(), True),
    StructField("VALOR_PROPOSTA", DoubleType(), True),
    StructField("VALOR_FINANCIAMENTO", DoubleType(), True),
    StructField("VALOR_ENTRADA", DoubleType(), True),
    StructField("VALOR_PRESTACAO", DoubleType(), True),
    StructField("QUANTIDADE_PARCELAS", IntegerType(), True),
    StructField("CARENCIA", IntegerType(), True),
    StructField("STATUS_PROPOSTA", StringType(), True)
])

schema_fct_transacoes = StructType([
    StructField("PK_TRANSACAO", LongType(), True),
    StructField("FK_CONTA", LongType(), True),
    StructField("FK_CLIENTE", LongType(), True),
    StructField("FK_AGENCIA", LongType(), True),
    StructField("FK_COLABORADOR", LongType(), True),
    StructField("NUMERO_TRANSACAO", LongType(), True),
    StructField("TS_TRANSACAO", TimestampType(), True),
    StructField("NOME_TRANSACAO", StringType(), True),
    StructField("TIPO_TRASACAO", StringType(), True),
    StructField("VALOR_TRANSACAO", DoubleType(), True)
])

In [0]:
tables_and_schemas = {
  "agg_transacoes_diarias": schema_agg_transacoes_diarias,
  "dim_agencias": schema_dim_agencias,
  "dim_clientes": schema_dim_clientes,
  "dim_colaboradores": schema_dim_colaboradores,
  "dim_contas": schema_dim_contas,
  "dim_datas": schema_dim_datas,
  "fct_contas": schema_fct_contas,
  "fct_propostas_de_credito": schema_fct_propostas_de_credito,
  "fct_transacoes": schema_fct_transacoes,
}

In [0]:
def get_files_names(directory_name):
   """
   Gets a list of file names from a specified directory.

   Args:
      directory_name: The name of the directory to list files from.

   Returns:
      A list of cleaned file names without the .csv extension.
   """

   directory_path = f"./{directory_name}"

   files_list = os.listdir(directory_path)

   cleaned_files_list = []

   for file_name in files_list:
      file_name = file_name.removesuffix(".csv")
      cleaned_files_list.append(file_name)

   return cleaned_files_list

In [0]:
def get_pandas_loading_args(spark_schema):
    """
    Analyzes a Spark schema and returns a tuple with two items:
    1. A dictionary of dtypes for non-date columns.
    2. A list of column names that should be parsed as dates.
    """
    dtype_mapping = {
        StringType: 'object',
        LongType: 'int64',
        IntegerType: 'int64',
        DoubleType: 'float64',
        BooleanType: 'bool'
    }
    
    pandas_dtypes = {}
    date_columns = []
    
    for field in spark_schema.fields:
        spark_type = type(field.dataType)
        
        if spark_type in [DateType, TimestampType]:
            date_columns.append(field.name)
        else:
            pandas_dtypes[field.name] = dtype_mapping.get(spark_type, 'object')
            
    return (pandas_dtypes, date_columns)

In [0]:
def write_csvs_to_delta(directory_name, file_name):
    """
    Reads a CSV, converts it to a Spark DataFrame, and writes it as a Delta table.

    Args:
        directory_name: The directorycontaining the CSV file.
        file_name: The name of the file to be processed (without the .csv extension).
    """
    csv_path = f"./{directory_name}/{file_name}.csv"

    # In Databricks Free Edition, we don't have direct filesystem access to
    # read uploaded files with spark.read.csv.
    # We use pandas as a workaround to read the local file into a DataFrame first.
    dtypes_for_csv, dates_to_parse = get_pandas_loading_args(tables_and_schemas[file_name])

    pandas_df = pd.read_csv(
        csv_path,
        dtype=dtypes_for_csv,
        parse_dates=dates_to_parse
    )
    
    spark_df = spark.createDataFrame(data=pandas_df, schema=tables_and_schemas[file_name])
    
    spark_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(file_name)

In [0]:
def process_files_in_parallel(files_list, directory_name):
    """
    Processes a list of files in parallel using a thread pool.

    Uses functools.partial to pass the directory_name argument to the mapping function.

    Args:
        files_list: A list of file names to process.
        directory_name: The directory where the files are located.
    """
    num_threads = 1

    with ThreadPool(num_threads) as pool:
        task = partial(write_csvs_to_delta, directory_name)
        pool.map(task, files_list)

In [0]:
catalog_name="fada_academy"
directory_names=["banvic_marts"]

print(f"Ensuring the catalog {catalog_name} exists, then setting it as the current catalog for the session")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")

for directory in directory_names:
    print(f"Started processing {directory} files")

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{directory}")
    spark.sql(f"USE SCHEMA {directory}")

    files_list = get_files_names(directory)

    process_files_in_parallel(files_list, directory)

    print(f"Finished processing {directory} files")